In [ ]:
import os
os.environ.pop('CUDA_VISIBLE_DEVICES', None)
GPU_NUMBER = [9]
os.environ["CUDA_VISIBLE_DEVICES"] = ",".join([str(s) for s in GPU_NUMBER])

In [ ]:
import datasets
import numpy as np
import random
import pickle
import sys
sys.path.append('/home/wwd/codebox/ODT-main')
from src.train import unsupervised_train
model_type = "bert"
# max input size
max_input_size = 2**11  # 2048
# number of layers
num_layers = 6
# number of attention heads
num_attn_heads = 4
# number of embedding dimensions
num_embed_dim = 256
# intermediate size
intermed_size = num_embed_dim * 2
# activation function
activ_fn = "relu"
# initializer range, layer norm, dropout
initializer_range = 0.02
layer_norm_eps = 1e-12
attention_probs_dropout_prob = 0.02
hidden_dropout_prob = 0.018
# set training parameters
dataset_unlabell = datasets.load_from_disk('../dataSets/unlabel_cell.datasets')
num_examples = len(dataset_unlabell)
# number gpus
num_gpus = 1
# batch size for training and eval
geneformer_batch_size = 4
# max learning rate
max_lr = 1e-5
# learning schedule
lr_schedule_fn = "linear"
# warmup steps
warmup_steps = 10_000
# number of epochs
# epochs = 100
epochs = 1 # just for example
# optimizer
optimizer = "adamw"
# weight_decay
weight_decay = 0.001
with open("../dataSets/gene_tk.pkl", "rb") as fp:
    token_dictionary = pickle.load(fp)
def process_cell_genes(example):
    example['input_ids'] = [token_dictionary[gene] for gene in example['cell_genes']]
    random.shuffle(example['input_ids'])
    example['input_ids'] = example['input_ids'][:2048]
    return example
dataset_unlabell = dataset_unlabell.map(process_cell_genes, num_proc=12)
dataset_unlabell.save_to_disk('./dataSets/unlabel_cell_tk.datasets')
config_dict = {
    "seed_val" : 42,
    "seed_num" : np.random.seed(0),
    'max_input_size' : max_input_size,
    "hidden_size": num_embed_dim,
    "num_layers": num_layers,
    "initializer_range": initializer_range,
    "layer_norm_eps": layer_norm_eps,
    "attention_probs_dropout_prob": attention_probs_dropout_prob,
    "hidden_dropout_prob": hidden_dropout_prob,
    "intermediate_size": intermed_size,
    "hidden_act": activ_fn,
    "max_position_embeddings": max_input_size,
    "model_type": model_type,
    "num_attn_heads": num_attn_heads*2,
    'num_embed_dim': num_embed_dim,
    'num_examples': num_examples,
    'activ_fn': "relu",
    'batch_size': geneformer_batch_size,
    'max_learning_rate':1e-5,
    'lr_schedule_fn': lr_schedule_fn,
    'warmup_steps': warmup_steps,
    'epochs':epochs,
    'optimizer': optimizer,
    'weight_decay':weight_decay,
    "pad_token_id": token_dictionary.get("<pad>"),
    "vocab_size": len(token_dictionary),  # genes+2 for <mask> and <pad> tokens
}


unsupervised_train(model_path = "../models/geneformer", dataset_path='../dataSets/unlabel_cell_tk.datasets',tk_path='../dataSets/gene_tk.pkl',
                   p_num_path = '../dataSets/p_num.pkl', target_dir='../models/unlabel_model_example',config_dict = config_dict)